# ERGT 03 - Adaptive Relational Control

This notebook is the first Colab-facing wrapper for the open adaptive relational control path. It does not replace ERGT-01 or ERGT-02. Its job is to prove that the adaptive controller, live 100-step display, fail-fast safety path, and lightweight report bundle can run safely before any long 2000-step experiment.

Default output bundle: `ergt_03_adaptive_control_report_bundle.zip`

Default local review path after download: `C:\Users\Administrator\Downloads\ergt_03_adaptive_control_report_bundle.zip`


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

try:
    from IPython.display import Markdown, display
except Exception:  # pragma: no cover - notebook convenience only
    Markdown = None
    display = None

RUN_PROFILE = "adaptive_smoke"  # "adaptive_smoke" or "adaptive_2000_guarded"
DEVICE = "cuda"

AUTO_SHUTDOWN_COLAB_RUNTIME = True
AUTO_SHUTDOWN_AFTER_SUCCESS = True
AUTO_SHUTDOWN_ON_FAILURE = True
AUTO_SHUTDOWN_DELAY_SECONDS = 30

REPORT_BUNDLE_NAME = "ergt_03_adaptive_control_report_bundle.zip"
DEFAULT_LOCAL_REVIEW_PATH = r"C:\Users\Administrator\Downloads\ergt_03_adaptive_control_report_bundle.zip"

PROFILE_SETTINGS = {
    "adaptive_smoke": {
        "max_synthetic_steps": 300,
        "live_display_interval": 100,
        "run_contract_tests": True,
        "allow_training": False,
    },
    "adaptive_2000_guarded": {
        "max_synthetic_steps": 2000,
        "live_display_interval": 100,
        "run_contract_tests": True,
        "allow_training": False,
    },
}

if RUN_PROFILE not in PROFILE_SETTINGS:
    raise ValueError(f"Unknown RUN_PROFILE: {RUN_PROFILE}")

PROFILE = PROFILE_SETTINGS[RUN_PROFILE]
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "experiments").exists() and Path("/content/ERGT").exists():
    PROJECT_ROOT = Path("/content/ERGT")
    os.chdir(PROJECT_ROOT)

if not (PROJECT_ROOT / "experiments").exists():
    raise RuntimeError(
        "Open this notebook from the ERGT repository root, or clone ERGT to /content/ERGT first."
    )

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
RUN_ROOT = PROJECT_ROOT / "runs" / "notebook_03_adaptive_relational_control" / f"{timestamp}_{RUN_PROFILE}"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Run root: {RUN_ROOT}")
print(f"Profile: {RUN_PROFILE}")
print(f"Fixed bundle name: {REPORT_BUNDLE_NAME}")
print(f"Default local review path: {DEFAULT_LOCAL_REVIEW_PATH}")


In [ ]:
def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore  # noqa: F401

        return True
    except Exception:
        return False


def write_json(path: Path, payload: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, sort_keys=True) + "\n")


def run_cmd(cmd: list[str], *, timeout_sec: int = 600, check: bool = True) -> dict:
    started = time.perf_counter()
    print("\n$ " + " ".join(cmd), flush=True)
    completed = subprocess.run(
        cmd,
        cwd=PROJECT_ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout_sec,
    )
    if completed.stdout:
        print(completed.stdout, end="", flush=True)
    elapsed = time.perf_counter() - started
    print(f"returncode={completed.returncode}, elapsed={elapsed / 60:.2f} min", flush=True)
    record = {
        "cmd": cmd,
        "returncode": completed.returncode,
        "elapsed_seconds": elapsed,
        "stdout_tail": completed.stdout[-4000:],
    }
    if check and completed.returncode != 0:
        raise RuntimeError(f"Command failed with code {completed.returncode}: {cmd}")
    return record


LIGHTWEIGHT_SUFFIXES = {".json", ".jsonl", ".md", ".txt", ".csv", ".png"}
EXCLUDED_ARTIFACT_PATTERNS = ["checkpoints/", "*.pt", "*.pth", "*.ckpt", "optimizer_state*"]
MAX_BUNDLE_FILE_BYTES = 8 * 1024 * 1024


def _bundle_candidates() -> list[Path]:
    candidates = [RUN_ROOT]
    contracts = PROJECT_ROOT / "runs" / "contracts"
    for name in [
        "open_adaptive_relational_control_trainer.json",
        "live_100_step_diagnostic_table.json",
        "unified_telemetry_schema.json",
        "adaptive_notebook_ergt_03.json",
    ]:
        path = contracts / name
        if path.exists():
            candidates.append(path)
    return candidates


def _include_in_bundle(path: Path) -> bool:
    if not path.is_file():
        return False
    parts = set(path.parts)
    if "checkpoints" in parts or "__pycache__" in parts:
        return False
    if path.suffix.lower() not in LIGHTWEIGHT_SUFFIXES:
        return False
    if path.stat().st_size > MAX_BUNDLE_FILE_BYTES:
        return False
    return True


def export_report_bundle(reason: str) -> Path:
    manifest = {
        "reason": reason,
        "run_profile": RUN_PROFILE,
        "run_root": str(RUN_ROOT),
        "bundle_name": REPORT_BUNDLE_NAME,
        "default_local_review_path": DEFAULT_LOCAL_REVIEW_PATH,
        "excluded_artifact_patterns": EXCLUDED_ARTIFACT_PATTERNS,
        "lightweight_only": True,
    }
    write_json(RUN_ROOT / "bundle_manifest.json", manifest)
    bundle_path = (Path("/content") if running_in_colab() else RUN_ROOT) / REPORT_BUNDLE_NAME
    with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for candidate in _bundle_candidates():
            if candidate.is_file() and _include_in_bundle(candidate):
                archive.write(candidate, candidate.relative_to(PROJECT_ROOT).as_posix())
            elif candidate.is_dir():
                for path in candidate.rglob("*"):
                    if _include_in_bundle(path):
                        archive.write(path, path.relative_to(PROJECT_ROOT).as_posix())
    print(f"\nBundle ready: {bundle_path}")
    print(f"Default local review path after download: {DEFAULT_LOCAL_REVIEW_PATH}")
    if running_in_colab():
        try:
            from google.colab import files  # type: ignore

            files.download(str(bundle_path))
        except Exception as exc:
            print(f"Automatic download failed: {exc}")
    return bundle_path


def shutdown_colab_runtime_if_requested(reason: str, *, failed: bool = False) -> None:
    should_shutdown = AUTO_SHUTDOWN_COLAB_RUNTIME and (
        AUTO_SHUTDOWN_AFTER_SUCCESS or (failed and AUTO_SHUTDOWN_ON_FAILURE)
    )
    if not should_shutdown or not running_in_colab():
        return
    print(
        f"Colab runtime shutdown requested after {AUTO_SHUTDOWN_DELAY_SECONDS}s. Reason: {reason}",
        flush=True,
    )
    time.sleep(AUTO_SHUTDOWN_DELAY_SECONDS)
    from google.colab import runtime  # type: ignore

    runtime.unassign()


write_json(
    RUN_ROOT / "notebook_runtime_config.json",
    {
        "run_profile": RUN_PROFILE,
        "profile": PROFILE,
        "device": DEVICE,
        "auto_shutdown_colab_runtime": AUTO_SHUTDOWN_COLAB_RUNTIME,
        "auto_shutdown_after_success": AUTO_SHUTDOWN_AFTER_SUCCESS,
        "auto_shutdown_on_failure": AUTO_SHUTDOWN_ON_FAILURE,
        "report_bundle_name": REPORT_BUNDLE_NAME,
        "default_local_review_path": DEFAULT_LOCAL_REVIEW_PATH,
    },
)


## 1. Contract Preflight

This cell validates the already-built mechanics before any longer run is allowed. If it fails, the notebook exports the lightweight bundle and stops.

In [ ]:
try:
    preflight_records = []
    if PROFILE["run_contract_tests"]:
        preflight_records.append(
            run_cmd(
                [
                    sys.executable,
                    "-m",
                    "pytest",
                    "tests/test_open_adaptive_relational_control_trainer.py",
                    "tests/test_live_100_step_diagnostic_table.py",
                    "tests/test_unified_telemetry_schema.py",
                ],
                timeout_sec=10 * 60,
            )
        )
    for script in [
        "experiments/create_open_adaptive_relational_control_trainer_report.py",
        "experiments/create_live_100_step_diagnostic_table_report.py",
        "experiments/create_unified_telemetry_schema_report.py",
    ]:
        preflight_records.append(run_cmd([sys.executable, script], timeout_sec=5 * 60))
    write_json(RUN_ROOT / "preflight_records.json", preflight_records)
except BaseException:
    export_report_bundle(reason="preflight_failure")
    shutdown_colab_runtime_if_requested("preflight_failure", failed=True)
    raise


## 2. Adaptive Trainer Smoke

This smoke uses synthetic, schema-compatible rows so the notebook can verify live 100-step tables and plot payloads without burning a long GPU run.

In [ ]:
from experiments.open_adaptive_relational_control_trainer import (
    OpenAdaptiveTrainerConfig,
    run_open_adaptive_control_trainer,
)


def synthetic_adaptive_rows(max_step: int) -> list[dict]:
    conditions = [
        ("baseline", 0.00),
        ("alpha_zero", 0.01),
        ("real_memory_d", 0.14),
        ("random_memory_d", 0.07),
        ("shuffled_memory_d", 0.06),
    ]
    rows: list[dict] = []
    for condition, offset in conditions:
        for step in range(100, max_step + 1, 100):
            baseline_loss = 6.15 - 0.0017 * step
            alpha = min(0.08, 0.01 + 0.00004 * step) if condition != "baseline" else 0.0
            rows.append(
                {
                    "condition": condition,
                    "step": step,
                    "train_loss": baseline_loss - offset - 0.03,
                    "validation_loss": baseline_loss - offset,
                    "real_vs_baseline_delta": offset if condition == "real_memory_d" else None,
                    "loss_slope_gain": 0.02 + offset / 4 if condition == "real_memory_d" else 0.0,
                    "alpha": alpha,
                    "alpha_effective": alpha,
                    "geo_to_qk_ratio": 0.02 + alpha * 0.8 if condition != "baseline" else 0.0,
                    "memory_stability": 0.42 + offset,
                    "memory_turnover": max(0.05, 0.30 - offset),
                    "memory_persistence": 0.45 + offset,
                    "memory_rigidity": 0.10,
                    "noise_risk": max(0.04, 0.16 - offset),
                    "attention_behavior_regime": "useful_noncollapsed",
                    "attention_behavior_separation": offset,
                    "distance_contrast_retention": 0.60 + offset,
                    "future_leak_score": 0.0,
                    "meta_top_signal": "memory_state" if condition == "real_memory_d" else "loss_trend",
                    "meta_attention_entropy": 1.8,
                    "meta_alpha_weight": 0.20,
                    "meta_memory_weight": 0.25,
                    "meta_gate_weight": 0.12,
                    "meta_reach_weight": 0.17,
                    "meta_norm_weight": 0.18,
                    "controller_conflict_score": 0.12,
                    "meta_control_confidence": 0.74,
                    "alpha_decision": "grow_alpha" if condition == "real_memory_d" else "observe",
                    "memory_eta_decision": "increase_eta" if condition == "real_memory_d" else "hold",
                    "memory_decay_decision": "hold",
                    "gate_floor_decision": "hold",
                    "causal_reachability_decision": "hold",
                    "distance_norm_scale_decision": "hold",
                    "joint_budget_decision": "balanced",
                }
            )
    return rows


try:
    trainer_config = OpenAdaptiveTrainerConfig(
        live_display_interval=PROFILE["live_display_interval"],
        artifact_bundle_name=REPORT_BUNDLE_NAME,
    )
    trainer_summary = run_open_adaptive_control_trainer(
        synthetic_adaptive_rows(PROFILE["max_synthetic_steps"]),
        config=trainer_config,
    )
    if trainer_summary["trainer_status"] != "completed":
        raise RuntimeError(f"Adaptive trainer smoke did not complete: {trainer_summary['trainer_status']}")
    if not trainer_summary["live_diagnostic_rows"]:
        raise RuntimeError("No live_diagnostic_rows were emitted.")
    write_json(RUN_ROOT / "trainer_summary.json", trainer_summary)
    write_jsonl(RUN_ROOT / "progress_log.jsonl", trainer_summary["progress_log"])
    write_jsonl(RUN_ROOT / "controller_decision_log.jsonl", trainer_summary["controller_decision_log"])
    write_jsonl(RUN_ROOT / "meta_control_observer_log.jsonl", trainer_summary["meta_control_observer_log"])
    write_jsonl(RUN_ROOT / "control_separation_log.jsonl", trainer_summary["control_separation_log"])
    write_jsonl(RUN_ROOT / "safety_log.jsonl", trainer_summary["safety_log"])
    write_jsonl(RUN_ROOT / "live_diagnostic_rows.jsonl", trainer_summary["live_diagnostic_rows"])
    (RUN_ROOT / "live_diagnostic_tables.md").write_text(
        "\n\n".join(trainer_summary["live_diagnostic_tables"]) + "\n",
        encoding="utf-8",
    )
    write_json(RUN_ROOT / "live_diagnostic_plot_payloads.json", trainer_summary["live_diagnostic_plot_payloads"])
    print("Latest live 100-step diagnostic table:")
    latest_table = trainer_summary["live_diagnostic_tables"][-1]
    if Markdown is not None and display is not None:
        display(Markdown(latest_table))
    else:
        print(latest_table)
except BaseException:
    export_report_bundle(reason="adaptive_trainer_smoke_failure")
    shutdown_colab_runtime_if_requested("adaptive_trainer_smoke_failure", failed=True)
    raise


## 3. Live Plots

The table is primary during a run. These plots are lightweight previews from the same plot payloads that go into the bundle.

In [ ]:
try:
    import matplotlib.pyplot as plt

    latest_payload = trainer_summary["live_diagnostic_plot_payloads"][-1]
    rows = trainer_summary["live_diagnostic_rows"]
    for key in ["validation_loss", "geo_to_qk_ratio", "memory_stability", "meta_memory_weight"]:
        series = [(row["step"], row.get(key), row["condition"]) for row in rows if row.get(key) is not None]
        if not series:
            continue
        plt.figure(figsize=(8, 3))
        for condition in sorted({item[2] for item in series}):
            xs = [step for step, _, cond in series if cond == condition]
            ys = [value for _, value, cond in series if cond == condition]
            plt.plot(xs, ys, marker="o", label=condition)
        plt.title(key)
        plt.xlabel(latest_payload.get("x_axis", "step"))
        plt.grid(True, alpha=0.25)
        plt.legend(loc="best")
        plt.tight_layout()
        plt.savefig(RUN_ROOT / f"live_preview_{key}.png", dpi=120)
        plt.show()
except Exception as exc:
    print(f"Plot preview skipped: {exc}")


## 4. Fail-Fast Smoke

The notebook must stop early when a hard safety condition appears. This cell verifies that a future-leak signal is reported immediately.

In [ ]:
try:
    fail_fast_rows = synthetic_adaptive_rows(200)[:2]
    fail_fast_rows.append(
        {
            **synthetic_adaptive_rows(300)[2],
            "condition": "real_memory_d",
            "step": 250,
            "future_leak_score": 0.2,
            "hard_stop_reason": "future_leak_score",
        }
    )
    fail_fast_rows.append({**synthetic_adaptive_rows(300)[3], "condition": "real_memory_d", "step": 300})
    fail_fast_report = run_open_adaptive_control_trainer(
        fail_fast_rows,
        config=OpenAdaptiveTrainerConfig(
            live_display_interval=PROFILE["live_display_interval"],
            artifact_bundle_name=REPORT_BUNDLE_NAME,
        ),
    )
    write_json(RUN_ROOT / "fail_fast_report.json", fail_fast_report)
    if fail_fast_report["trainer_status"] != "failed_fast":
        raise RuntimeError("Fail-fast smoke did not trigger on future_leak_score.")
    if fail_fast_report["trainer_processed_rows"] >= len(fail_fast_rows):
        raise RuntimeError("Fail-fast smoke did not stop before later rows.")
    print("Fail-fast status:", fail_fast_report["trainer_status"])
    print("Fail-fast reason:", fail_fast_report["trainer_fail_fast_reason"])
    if Markdown is not None and display is not None:
        display(Markdown(fail_fast_report["live_diagnostic_tables"][-1]))
except BaseException:
    export_report_bundle(reason="fail_fast_smoke_failure")
    shutdown_colab_runtime_if_requested("fail_fast_smoke_failure", failed=True)
    raise


## 5. Export and Shutdown

The bundle intentionally excludes checkpoints and large model artifacts. It contains configs, metrics, progress logs, controller logs, live diagnostic rows/tables/plot payloads, fail-fast report, and contract summaries.

In [ ]:
try:
    stage19_summary = {
        "status": "pass",
        "run_profile": RUN_PROFILE,
        "trainer_status": trainer_summary["trainer_status"],
        "fail_fast_status": fail_fast_report["trainer_status"],
        "live_diagnostic_row_count": trainer_summary["live_diagnostic_row_count"],
        "live_diagnostic_table_ready": trainer_summary["live_diagnostic_table_ready"],
        "live_diagnostic_plot_ready": trainer_summary["live_diagnostic_plot_ready"],
        "artifact_bundle_name": REPORT_BUNDLE_NAME,
        "default_local_review_path": DEFAULT_LOCAL_REVIEW_PATH,
        "checkpoint_artifacts_excluded": True,
        "next_required_step": "short_smoke_and_failure_safety_validation",
    }
    write_json(RUN_ROOT / "stage19_notebook_summary.json", stage19_summary)
    bundle_path = export_report_bundle(reason="completed")
    print("\nDecision:", stage19_summary["next_required_step"])
    print("Fixed output bundle name:", REPORT_BUNDLE_NAME)
    print("Default local review path after Colab download:", DEFAULT_LOCAL_REVIEW_PATH)
    shutdown_colab_runtime_if_requested("completed", failed=False)
except BaseException:
    export_report_bundle(reason="final_export_failure")
    shutdown_colab_runtime_if_requested("final_export_failure", failed=True)
    raise
